### Data Transformation

In [19]:
import os
%pwd

'/workspaces/summarizer'

In [20]:
os.chdir("textsummariser")

In [21]:
%pwd

'/workspaces/summarizer/textsummariser'

In [22]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [23]:
from src.textSummariser.constants import *
from src.textSummariser.utils.common import read_yaml, create_directories

In [24]:
class ConfigurationManager:
    def __init__(self,
                config_path = CONFIG_FILE_PATH,
                params_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = Path(config.root_dir),
            data_path = Path(config.data_path),
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

In [25]:
import os
from src.textSummariser.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk

## DataTransformation Components

In [26]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)

    def convert_examples_to_features(self, example_batch): # Added self here!
        
        input_encodings = self.tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)

        target_encodings = self.tokenizer(text_target=example_batch['summary'], max_length=128, truncation=True)

        return {
            'input_ids' : input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        
        # FIXED: Glued the paths together into one string
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))

In [27]:
config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.convert()


[2026-09-26 08:50:03,288: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-09-26 08:50:03,294: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-26 08:50:03,295: INFO: common: created directory at: artifacts]
[2026-09-26 08:50:03,296: INFO: common: created directory at: artifacts/data_transformation]
[2026-09-26 08:50:03,420: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-09-26 08:50:03,427: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2Fmain%2Fconfig.json=&etag=%222c1a911e577525af99c26c1634c473667e1e7ae2%22 "HTTP/1.1 200 OK"]
[2026-09-26 08:50:03,572: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/14732 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/819 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/818 [00:00<?, ? examples/s]